In [39]:
import pandas as pd
import numpy as np
import json
import os
import ast
from shapely.geometry import Polygon

In [40]:
import json
import os

parsed_dir = "parsed_floorplans"

parsed_plans = {}

for f in os.listdir(parsed_dir):

    if f.endswith(".json"):

        plan_id = f.replace(".json","")

        with open(os.path.join(parsed_dir,f)) as file:
            parsed_plans[plan_id] = json.load(file)

In [41]:
def polygon_from_points(points):

    if isinstance(points, str):
        points = ast.literal_eval(points)

    return Polygon(points)

In [ ]:
CIRCULATION_KEYWORDS = [
    "corridor",
    "hall",
    "hallway",
    "lobby",
    "entry",
    "foyer",
    "passage",
    "stair"
]

def compute_spatial_efficiency(plan_rooms):

    total_area = plan_rooms["area"].sum()

    circulation_mask = plan_rooms["room_type"].str.lower().apply(
        lambda x: any(k in x for k in CIRCULATION_KEYWORDS)
    )

    circulation_rooms = plan_rooms[circulation_mask]

    circulation_area = circulation_rooms["area"].sum()

    if total_area == 0:
        return 0, 0

    efficiency = (total_area - circulation_area) / total_area
    
    corridor_ratio = circulation_area / total_area

    return efficiency, corridor_ratio

In [ ]:
def compute_room_area_stats(plan_rooms):

    avg_room_area = plan_rooms["area"].mean()

    std_room_area = plan_rooms["area"].std()

    return avg_room_area, std_room_area

In [44]:
import os
import json

parsed_folder = "parsed_floorplans"

plan_geometry = {}

for file in os.listdir(parsed_folder):

    if file.endswith(".json"):

        plan_id = file.replace(".json", "")

        with open(os.path.join(parsed_folder, file)) as f:
            plan_geometry[plan_id] = json.load(f)

print("Loaded geometry for", len(plan_geometry), "plans")

Loaded geometry for 4925 plans


In [45]:
from shapely.geometry import Polygon

def polygon_area(coords):

    if coords is None or len(coords) < 3:
        return 0

    try:
        poly = Polygon(coords)

        if not poly.is_valid:
            return 0

        return poly.area

    except:
        return 0

In [3]:
def compute_environmental_features(plan_id, plan_rooms):

    plan_id = str(plan_id)

    if plan_id not in plan_geometry:
        return 0, 0, 0, 0

    geom = plan_geometry[plan_id]

    windows = geom["windows"]
    walls = geom["walls"]

    # --------------------------------------------------
    # 1. Window Ratio (total window length / wall length)
    # --------------------------------------------------
    window_length = 0

    for w in windows:
        if len(w) < 2:
            continue

        p1 = w[0]
        p2 = w[1]

        length = np.sqrt((p2[0]-p1[0])**2 + (p2[1]-p1[1])**2)
        window_length += length


    wall_length = 0

    for wall in walls:
        pts = wall["polygon"]

        for i in range(len(pts)):
            x1, y1 = pts[i]
            x2, y2 = pts[(i+1) % len(pts)]

            wall_length += np.sqrt((x2-x1)**2 + (y2-y1)**2)


    window_ratio = window_length / wall_length if wall_length > 0 else 0
    window_ratio = min(window_ratio, 1)


    # --------------------------------------------------
    # 2. Rooms Without Windows
    # --------------------------------------------------
    rooms_without_window = 0

    for _, room in plan_rooms.iterrows():

        room_poly = polygon_from_points(room["points"])

        has_window = False

        for w in windows:

            if len(w) < 2:
                continue

            window_line = LineString(w)

            if room_poly.buffer(5).intersects(window_line):
                has_window = True
                break

        if not has_window:
            rooms_without_window += 1


    rooms_without_window_ratio = (
        rooms_without_window / len(plan_rooms)
        if len(plan_rooms) > 0 else 0
    )


    # --------------------------------------------------
    # 3. External Wall Exposure
    # --------------------------------------------------
    external_walls = [w for w in walls if w["type"] == "external"]

    external_wall_exposure = (
        len(external_walls) / len(walls)
        if len(walls) > 0 else 0
    )


    # --------------------------------------------------
    # 4. Cross Ventilation Detection
    # --------------------------------------------------
    rooms_with_cross_vent = 0

    for _, room in plan_rooms.iterrows():

        room_poly = polygon_from_points(room["points"])

        window_angles = []

        for w in windows:

            if len(w) < 2:
                continue

            p1 = w[0]
            p2 = w[1]

            window_line = LineString(w)

            if room_poly.buffer(5).intersects(window_line):

                dx = p2[0] - p1[0]
                dy = p2[1] - p1[1]

                angle = abs(np.arctan2(dy, dx))
                window_angles.append(angle)


        # check if any pair of windows are opposite
        has_cross_vent = False

        for i in range(len(window_angles)):
            for j in range(i+1, len(window_angles)):

                if abs(window_angles[i] - window_angles[j]) > 2.5:
                    has_cross_vent = True
                    break

            if has_cross_vent:
                break


        if has_cross_vent:
            rooms_with_cross_vent += 1


    cross_ventilation_ratio = (
        rooms_with_cross_vent / len(plan_rooms)
        if len(plan_rooms) > 0 else 0
    )


    return (
        window_ratio,
        rooms_without_window_ratio,
        external_wall_exposure,
        cross_ventilation_ratio
    )

In [62]:
from shapely.geometry import Polygon, LineString
def build_plan_level_features(rooms_df):

    features = []

    for plan_id in rooms_df["plan_id"].unique():

        plan_rooms = rooms_df[rooms_df["plan_id"] == plan_id]

        efficiency, corridor_ratio = compute_spatial_efficiency(plan_rooms)

        avg_room_area, std_room_area = compute_room_area_stats(plan_rooms)

        (
            window_ratio,
            rooms_without_window,
            external_wall_exposure,
            cross_ventilation_ratio
        ) = compute_environmental_features(plan_id, plan_rooms)

        features.append({
            "plan_id": plan_id,
            "num_rooms": len(plan_rooms),
            "total_area": plan_rooms["area"].sum(),
            "efficiency": efficiency,
            "corridor_ratio": corridor_ratio,
            "avg_room_area": avg_room_area,
            "std_room_area": std_room_area,
            "window_ratio": window_ratio,
            "rooms_without_window_ratio": rooms_without_window,
            "external_wall_exposure": external_wall_exposure,
            "cross_ventilation_ratio": cross_ventilation_ratio
        })

    return pd.DataFrame(features)

In [63]:
rooms_df = pd.read_csv("rooms_dataset.csv")

rooms_df.head()

,room_type,area,centroid_x,centroid_y,num_vertices,points,plan_id
0,Outdoor Terrace,248496.0296,1260.503333,387.083333,6,"[(1030.81, 507.29), (1030.81, 146.67), (1719.8...",6044
1,Entry Lobby,54825.5028,1414.466667,1211.093333,9,"[(1469.34, 1137.48), (1469.34, 1148.48), (1530...",6044
2,Kitchen,67064.8976,1576.103333,1320.606667,6,"[(1712.49, 1478.05), (1474.05, 1478.05), (1474...",6044
3,LivingRoom,221476.7801,1482.916667,934.763333,6,"[(1712.49, 1137.48), (1469.34, 1137.48), (1334...",6044
4,Utility Laundry,20826.7515,1121.873333,1377.933333,6,"[(1190.03, 1478.05), (1051.17, 1478.05), (1051...",6044


In [64]:
plan_features = build_plan_level_features(rooms_df)

plan_features.head()

,plan_id,num_rooms,total_area,efficiency,corridor_ratio,avg_room_area,std_room_area,window_ratio,rooms_without_window_ratio,external_wall_exposure,cross_ventilation_ratio
0,6044,10,8.246375e+05,0.911140,0.088860,82463.748340,85186.291774,0.044746,0.400000,0.285714,0.000000
1,2564,24,5.258796e+06,1.000000,0.000000,219116.517508,316108.739301,0.073258,0.500000,0.260870,0.166667
2,6165,6,4.252975e+05,1.000000,0.000000,70882.923650,69961.339122,0.080980,0.166667,0.260870,0.000000
3,1021,17,2.011410e+06,0.960816,0.039184,118318.259526,146612.496741,0.035671,0.470588,0.743243,0.058824
4,5507,16,9.967282e+05,0.895836,0.104164,62295.510366,41325.183411,0.057300,0.375000,0.500000,0.000000


In [65]:
plan_features.to_csv("plan_level_features.csv", index=False)

In [ ]:
from shapely.geometry import Polygon
import numpy as np

def compute_window_ratio(plan_id):

    plan = parsed_plans.get(str(plan_id))

    if plan is None:
        return 0

    windows = plan["windows"]
    walls = plan["walls"]

    window_area = sum(Polygon(w).area for w in windows if len(w) >= 3)

    wall_length = 0

    for wall in walls:

        pts = wall["polygon"]

        for i in range(len(pts)):

            x1,y1 = pts[i]
            x2,y2 = pts[(i+1)%len(pts)]

            wall_length += np.sqrt((x2-x1)**2 + (y2-y1)**2)

    return window_area / wall_length if wall_length > 0 else 0

In [8]:
# ==========================================
# PLAN LEVEL FEATURE EXTRACTION
# ==========================================

import json
import os
import numpy as np
import pandas as pd

parsed_dir = "parsed_floorplans"


# -----------------------------------------
# Room categories
# -----------------------------------------

circulation_keywords = [
    "corridor",
    "hall",
    "hallway",
    "lobby",
    "entry",
    "entrance",
    "foyer",
    "passage"
]

service_types = [
    "bathroom",
    "toilet",
    "storage"
]


# -----------------------------------------
# Feature Computation
# -----------------------------------------

records = []

for file in os.listdir(parsed_dir):

    path = os.path.join(parsed_dir, file)

    with open(path) as f:
        data = json.load(f)

    rooms = data["rooms"]
    windows = data["windows"]
    walls = data["walls"]

    plan_id = file.replace(".json","")

    room_areas = [r["area"] for r in rooms]

    total_area = sum(room_areas)

    circulation_area = sum(
        r["area"] for r in rooms
        if any(k in r["type"] for k in circulation_keywords)
    )

    usable_area = total_area - circulation_area

    efficiency = usable_area / total_area if total_area else 0

    corridor_ratio = circulation_area / total_area if total_area else 0

    if len(room_areas) > 0:
        avg_room_area = np.mean(room_areas)
        std_room_area = np.std(room_areas)
    else:
        avg_room_area = 0
        std_room_area = 0

    # ----------------------------------
    # Window statistics
    # ----------------------------------

    rooms_with_window = 0

    for r in rooms:

        cx, cy = r["centroid"]

        for w in windows:

            wx, wy = w["centroid"]

            dist = np.linalg.norm(
                np.array([cx,cy]) - np.array([wx,wy])
            )

            if dist < 250:

                rooms_with_window += 1
                break

    window_room_ratio = rooms_with_window / len(rooms) if rooms else 0

    # ----------------------------------
    # External walls
    # ----------------------------------

    external_walls = [w for w in walls if w["type"] == "external"]

    external_wall_ratio = len(external_walls) / len(walls) if walls else 0

    # ----------------------------------
    # Cross ventilation
    # ----------------------------------

    cross_vent_rooms = 0

    for r in rooms:

        nearby_windows = []

        cx, cy = r["centroid"]

        for w in windows:

            wx, wy = w["centroid"]

            if np.linalg.norm(np.array([cx,cy]) - np.array([wx,wy])) < 250:

                nearby_windows.append((wx,wy))

        if len(nearby_windows) >= 2:

            xs = [w[0] for w in nearby_windows]

            if max(xs) - min(xs) > 2:
                cross_vent_rooms += 1
    cross_vent_ratio = cross_vent_rooms / len(rooms) if rooms else 0

    # ----------------------------------

    external_walls = [w for w in walls if w["type"] == "external"]
    total_window_area = sum(w["area"] for w in windows)

    window_wall_ratio = total_window_area / total_area if total_area else 0
    records.append({

        "plan_id": plan_id,

        "num_rooms": len(rooms),

        "gross_area": total_area,

        "usable_area": usable_area,

        "efficiency": efficiency,

        "corridor_ratio": corridor_ratio,

        "avg_room_area": avg_room_area,

        "std_room_area": std_room_area,

        "rooms_with_window_ratio": window_room_ratio,

        "external_wall_ratio": external_wall_ratio,

        "cross_ventilation_ratio": cross_vent_ratio,
        "window_wall_ratio": window_wall_ratio,
    })


df = pd.DataFrame(records)

df.head()

df.to_csv("plan_level_features.csv", index=False)